# LLM Reasoning Fine-Tuning Pipeline

End-to-end pipeline: classify puzzles → solve → generate traces → format SFT → QLoRA train → evaluate → submit

**Requirements**: Kaggle GPU (RTX PRO 6000, 96GB VRAM)

## 0. Setup

In [ ]:
%%bash
# Clone the repo (or upload as dataset)
# Uncomment and set your repo URL:
# git clone https://github.com/YOUR_USER/homework3_llm_reasoning_finetuning.git /kaggle/working/repo

# If code is already in /kaggle/working, skip the clone
pip install -q peft>=0.12.0 trl>=0.12.0 bitsandbytes>=0.44.0 accelerate>=1.0.0 pyyaml polars

In [ ]:
import os, sys, json

# Set working directory to repo root
REPO_ROOT = '/kaggle/working'  # adjust if cloned elsewhere
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

# Ensure directories exist
for d in ['data/splits', 'data/traces', 'data/sft', 'data/solver_results',
          'experiments/configs', 'experiments/results',
          'checkpoints', 'submissions', 'notebooks']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'GPU: {os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").read().strip()}')

## 1. Data Preparation

In [ ]:
# Copy competition data
import shutil
comp_data = '/kaggle/input/nvidia-nemotron-model-reasoning-challenge'
if os.path.exists(comp_data):
    shutil.copy(f'{comp_data}/train.csv', 'data/train.csv')
    shutil.copy(f'{comp_data}/test.csv', 'data/test.csv')
    print('Copied competition data')
else:
    print('Competition data not found as input dataset. Make sure to add it.')

In [ ]:
# Classify puzzles
!python src/data/prepare.py --input data/train.csv --output data/puzzles_classified.jsonl

In [ ]:
# Create train/val split
!python src/data/splits.py --input data/puzzles_classified.jsonl --output data/splits/ --val-pct 0.10 --seed 42

## 2. Run Solvers & Verify

In [ ]:
import json
from src.metrics.competition import verify
from src.solvers.numeral import NumeralSolver
from src.solvers.gravity import GravitySolver
from src.solvers.unit_conversion import UnitConversionSolver
from src.solvers.cipher import CipherSolver
from src.solvers.bit_manipulation import BitManipulationSolver
from src.solvers.equation import EquationSolver
from src.solvers.cryptarithm import CryptarithmSolver

# Load puzzles
puzzles = []
with open('data/puzzles_classified.jsonl') as f:
    for line in f:
        puzzles.append(json.loads(line))

# Map categories to solvers
solvers = {
    'numeral': NumeralSolver(),
    'gravity': GravitySolver(),
    'unit_conversion': UnitConversionSolver(),
    'cipher': CipherSolver(),
    'bit_manipulation': BitManipulationSolver(),
    'equation_numeric_deduce': EquationSolver(),
    'equation_numeric_guess': EquationSolver(),
    'cryptarithm_deduce': CryptarithmSolver(),
    'cryptarithm_guess': CryptarithmSolver(),
}

# Run all solvers and collect results
all_results = []
cat_stats = {}

for p in puzzles:
    cat = p['category']
    solver = solvers.get(cat)
    if solver is None:
        continue
    result = solver.solve(p)
    correct = verify(p['answer'], result.predicted_answer)
    all_results.append({
        'puzzle_id': p['id'],
        'category': cat,
        'predicted_answer': result.predicted_answer,
        'expected_answer': p['answer'],
        'is_correct': correct,
        'solve_method': result.solve_method,
        'confidence': result.confidence,
    })
    cat_stats.setdefault(cat, [0, 0])
    cat_stats[cat][1] += 1
    if correct:
        cat_stats[cat][0] += 1

print(f"{'Category':<30} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print('-' * 60)
total_correct = total_all = 0
for cat in sorted(cat_stats):
    c, t = cat_stats[cat]
    total_correct += c
    total_all += t
    print(f"{cat:<30} {c:>8} {t:>8} {c/t:>10.4f}")
print('-' * 60)
print(f"{'TOTAL':<30} {total_correct:>8} {total_all:>8} {total_correct/total_all:>10.4f}")

## 3. Generate CoT Traces

In [ ]:
from src.trace_generators.numeral_traces import NumeralTraceGenerator
from src.trace_generators.gravity_traces import GravityTraceGenerator
from src.trace_generators.unit_conversion_traces import UnitConversionTraceGenerator
from src.trace_generators.cipher_traces import CipherTraceGenerator
from src.trace_generators.bit_manipulation_traces import BitManipulationTraceGenerator
from src.trace_generators.equation_traces import EquationTraceGenerator
from src.trace_generators.cryptarithm_traces import CryptarithmTraceGenerator

trace_generators = {
    'numeral': NumeralTraceGenerator(),
    'gravity': GravityTraceGenerator(),
    'unit_conversion': UnitConversionTraceGenerator(),
    'cipher': CipherTraceGenerator(),
    'bit_manipulation': BitManipulationTraceGenerator(),
    'equation_numeric_deduce': EquationTraceGenerator(),
    'equation_numeric_guess': EquationTraceGenerator(),
    'cryptarithm_deduce': CryptarithmTraceGenerator(),
    'cryptarithm_guess': CryptarithmTraceGenerator(),
}

# Build lookup: puzzle_id -> result
result_lookup = {r['puzzle_id']: r for r in all_results if r['is_correct']}
puzzle_lookup = {p['id']: p for p in puzzles}

# Generate traces for correctly solved puzzles
traces_by_cat = {}
for puzzle_id, r in result_lookup.items():
    cat = r['category']
    gen = trace_generators.get(cat)
    if gen is None:
        continue
    puzzle = puzzle_lookup[puzzle_id]
    solver = solvers[cat]
    solver_result = solver.solve(puzzle)
    try:
        trace = gen.generate_trace(puzzle, solver_result)
        if trace.final_answer:  # skip empty traces
            trace_dict = {
                'puzzle_id': trace.puzzle_id,
                'category': trace.category,
                'thinking_text': trace.thinking_text,
                'final_answer': trace.final_answer,
                'token_count': trace.token_count,
                'is_verified': trace.is_verified,
            }
            traces_by_cat.setdefault(cat, []).append(trace_dict)
    except Exception as e:
        pass  # Skip failed trace generation

# Save traces
total_traces = 0
for cat, traces in traces_by_cat.items():
    # Map subcategories to base category for filename
    base_cat = cat.replace('_deduce', '').replace('_guess', '').replace('_numeric', '')
    outfile = f'data/traces/{base_cat}_traces.jsonl'
    mode = 'a' if os.path.exists(outfile) else 'w'
    with open(outfile, mode) as f:
        for t in traces:
            f.write(json.dumps(t) + '\n')
    total_traces += len(traces)
    print(f'{cat}: {len(traces)} traces')

print(f'\nTotal traces generated: {total_traces}')

## 4. Format SFT Data

In [ ]:
from src.data.format_sft import format_trace_to_sft

# Load train split IDs
with open('data/splits/train_ids.json') as f:
    train_ids = set(json.load(f))

# Format all traces as SFT examples
sft_examples = []
for cat, traces in traces_by_cat.items():
    for t in traces:
        pid = t['puzzle_id']
        if str(pid) not in train_ids and pid not in train_ids:
            continue  # Only use train split
        if t['token_count'] > 7680:
            continue  # Skip too-long traces
        puzzle = puzzle_lookup.get(pid)
        if puzzle is None:
            continue
        sft = format_trace_to_sft(t, puzzle)
        sft_examples.append(sft)

# Save SFT data
with open('data/sft/train_sft.jsonl', 'w') as f:
    for ex in sft_examples:
        f.write(json.dumps(ex) + '\n')

print(f'SFT examples: {len(sft_examples)}')

# Per-category breakdown
from collections import Counter
cat_counts = Counter(ex['category'] for ex in sft_examples)
for cat, count in sorted(cat_counts.items()):
    print(f'  {cat}: {count}')

## 5. Baseline Evaluation (No Training)

In [ ]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

# Try kagglehub first (Kaggle environment), fall back to HF
try:
    import kagglehub
    MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
    print(f'Model path (kagglehub): {MODEL_PATH}')
except Exception:
    MODEL_PATH = MODEL_NAME
    print(f'Using HuggingFace model: {MODEL_PATH}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f'Tokenizer loaded: vocab_size={tokenizer.vocab_size}')

In [ ]:
from vllm import LLM, SamplingParams

# Load val split
with open('data/splits/val_ids.json') as f:
    val_ids = set(json.load(f))

val_puzzles = [p for p in puzzles if str(p['id']) in val_ids or p['id'] in val_ids]
print(f'Validation puzzles: {len(val_puzzles)}')

# Build prompts
prompts = []
for p in val_puzzles:
    user_content = p['prompt'] + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    prompts.append(prompt)

# Run baseline inference
llm = LLM(
    model=MODEL_PATH,
    max_model_len=8192,
    gpu_memory_utilization=0.85,
    trust_remote_code=True,
)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=7680,
)

outputs = llm.generate(prompts, sampling_params)
responses = [o.outputs[0].text for o in outputs]
print(f'Generated {len(responses)} responses')

In [ ]:
from src.metrics.competition import extract_final_answer, verify

# Score baseline
baseline_stats = {}
for p, resp in zip(val_puzzles, responses):
    cat = p['category']
    predicted = extract_final_answer(resp)
    correct = verify(p['answer'], predicted)
    baseline_stats.setdefault(cat, [0, 0])
    baseline_stats[cat][1] += 1
    if correct:
        baseline_stats[cat][0] += 1

print('=== BASELINE (no training) ===')
print(f"{'Category':<30} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print('-' * 60)
tc = ta = 0
for cat in sorted(baseline_stats):
    c, t = baseline_stats[cat]
    tc += c; ta += t
    print(f"{cat:<30} {c:>8} {t:>8} {c/t:>10.4f}")
print('-' * 60)
print(f"{'TOTAL':<30} {tc:>8} {ta:>8} {tc/ta:>10.4f}")

# Save baseline results
with open('experiments/results/exp-001-baseline.json', 'w') as f:
    json.dump({
        'overall_accuracy': tc/ta,
        'per_category_accuracy': {cat: c/t for cat, (c, t) in baseline_stats.items()},
        'per_category_counts': {cat: {'correct': c, 'total': t} for cat, (c, t) in baseline_stats.items()},
    }, f, indent=2)

# Clean up baseline model to free VRAM
del llm
import gc; gc.collect()
torch.cuda.empty_cache()

## 6. QLoRA Fine-Tuning

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Model loaded. Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Apply LoRA
import re as _re

# Find target modules matching the regex pattern
target_pattern = _re.compile(r'.*\.(in_proj|out_proj|up_proj|down_proj)$')
target_modules = list(set(
    name.split('.')[-1]
    for name, _ in model.named_modules()
    if target_pattern.match(name)
))
if not target_modules:
    target_modules = ['in_proj', 'out_proj', 'up_proj', 'down_proj']

print(f'Target modules: {target_modules}')

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load SFT data
sft_data = []
with open('data/sft/train_sft.jsonl') as f:
    for line in f:
        sft_data.append(json.loads(line))

# Convert to HF Dataset
# The SFT data has 'messages' format compatible with TRL
train_dataset = Dataset.from_list(sft_data)
print(f'Training examples: {len(train_dataset)}')
print(f'Sample keys: {list(sft_data[0].keys())}')

In [ ]:
# Train
OUTPUT_DIR = 'checkpoints/exp-full-sft'

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    max_seq_length=4096,
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    save_strategy='epoch',
    seed=42,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print('Starting training...')
train_result = trainer.train()
print(f'Training complete. Loss: {train_result.training_loss:.4f}')

# Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')

In [ ]:
# Free training model memory
del model, trainer
import gc; gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 7. Evaluate Fine-Tuned Model

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

# Load model with adapter for evaluation
llm = LLM(
    model=MODEL_PATH,
    max_model_len=8192,
    gpu_memory_utilization=0.85,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=32,
)

lora_request = LoRARequest('finetuned', 1, OUTPUT_DIR)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=7680,
)

# Run inference on val split
outputs = llm.generate(prompts, sampling_params, lora_request=lora_request)
ft_responses = [o.outputs[0].text for o in outputs]
print(f'Generated {len(ft_responses)} responses')

In [ ]:
# Score fine-tuned model
ft_stats = {}
for p, resp in zip(val_puzzles, ft_responses):
    cat = p['category']
    predicted = extract_final_answer(resp)
    correct = verify(p['answer'], predicted)
    ft_stats.setdefault(cat, [0, 0])
    ft_stats[cat][1] += 1
    if correct:
        ft_stats[cat][0] += 1

print('=== FINE-TUNED MODEL ===')
print(f"{'Category':<30} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print('-' * 60)
tc = ta = 0
for cat in sorted(ft_stats):
    c, t = ft_stats[cat]
    tc += c; ta += t
    # Show improvement vs baseline
    bl_acc = baseline_stats.get(cat, [0, 1])
    bl_pct = bl_acc[0]/bl_acc[1] if bl_acc[1] else 0
    delta = c/t - bl_pct
    print(f"{cat:<30} {c:>8} {t:>8} {c/t:>10.4f}  ({delta:+.4f})")
print('-' * 60)
bl_total = sum(v[0] for v in baseline_stats.values()) / sum(v[1] for v in baseline_stats.values())
print(f"{'TOTAL':<30} {tc:>8} {ta:>8} {tc/ta:>10.4f}  ({tc/ta - bl_total:+.4f})")

# Save results
with open('experiments/results/exp-full-sft.json', 'w') as f:
    json.dump({
        'overall_accuracy': tc/ta,
        'per_category_accuracy': {cat: c/t for cat, (c, t) in ft_stats.items()},
    }, f, indent=2)

del llm
gc.collect()
torch.cuda.empty_cache()

## 8. Package & Submit

In [ ]:
import subprocess

# Package adapter for submission
os.makedirs('submissions', exist_ok=True)
os.chdir(OUTPUT_DIR)
subprocess.run('zip -m /kaggle/working/submission.zip adapter_config.json adapter_model.safetensors',
               shell=True, check=True)
os.chdir(REPO_ROOT)

# Verify submission
import zipfile
with zipfile.ZipFile('/kaggle/working/submission.zip', 'r') as z:
    print('Submission contents:')
    for info in z.infolist():
        print(f'  {info.filename}: {info.file_size/1024:.1f} KB')

# Read adapter config
with z.open('adapter_config.json') as f:
    config = json.load(f)
    print(f'\nAdapter rank: {config.get("r", "unknown")}')
    print(f'Target modules: {config.get("target_modules", "unknown")}')

print('\nSubmission ready: /kaggle/working/submission.zip')